# SaProt with Real Protein Sequences (UniRef50)

Tests SaProt on **real protein sequences** from UniRef50.

## Purpose

- Tests ecological validity of SaProt's embedding geometry on real proteins
- Direct comparison with ESM-2 UniRef experiments
- 3 sizes (35M, 650M, 1.3B) for scaling analysis on real data

---

In [ ]:
# Install Dependencies
print("Installing core dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas einops datasets
import transformers, torch
print(f"transformers {transformers.__version__} | CUDA: {torch.cuda.is_available()}")
print("Done!")

In [ ]:
# Configuration
import sys, os
sys.path.insert(0, '.')
PHASE = 'full'
OUTPUT_BASE = './results/saprot_uniref_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'
CONFIG = {
    'quick': {'n_sequences': 500, 'min_length': 100, 'max_length': 400,
        'models': ['westlake-repl/SaProt_35M_AF2', 'westlake-repl/SaProt_650M_AF2'],
        'batch_size': 8, 'n_bootstrap': 0, 'snp_rates': [0.01, 0.05, 0.10]},
    'full': {'n_sequences': 10000, 'min_length': 100, 'max_length': 400,
        'models': [
            'westlake-repl/SaProt_35M_AF2',
            'westlake-repl/SaProt_650M_AF2',
            'westlake-repl/SaProt_1.3B_AFDB_OMG_NCBI',
        ],
        'batch_size': 8, 'n_bootstrap': 5, 'snp_rates': [0.01, 0.02, 0.05, 0.10],
        'batch_overrides': {'westlake-repl/SaProt_1.3B_AFDB_OMG_NCBI': 2}},
}
config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()} | Data: UniRef50 | Models: {len(config['models'])}")

In [ ]:
# Load Real Protein Sequences from UniRef50
import os
import numpy as np
VALID_AAS = set('ACDEFGHIKLMNPQRSTVWY')
MAX_SCAN_RECORDS = 200_000
def load_from_huggingface(n_sequences, min_len, max_len, seed=320):
    try:
        from datasets import load_dataset
        print("Loading UniRef50 from HuggingFace datasets...")
        ds = load_dataset('sagawa/uniref50-sample', split='train', streaming=True)
        rng = np.random.default_rng(seed)
        sequences = []; scanned = 0
        for record in ds:
            seq = record.get('sequence', record.get('text', ''))
            if not seq: continue
            scanned += 1
            if min_len <= len(seq) <= max_len and all(c in VALID_AAS for c in seq):
                sequences.append(seq)
            if scanned % 10000 == 0: print(f"  Scanned {scanned}, collected {len(sequences)}/{n_sequences}", end='\r')
            if len(sequences) >= n_sequences or scanned >= MAX_SCAN_RECORDS: break
        print(f"\nCollected {len(sequences)} sequences from HuggingFace")
        return sequences if len(sequences) >= n_sequences * 0.8 else None
    except Exception as e: print(f"HuggingFace loading failed: {e}"); return None
def generate_fallback_proteins(n_sequences, min_len, max_len, seed=320):
    rng = np.random.default_rng(seed)
    aas = list('ACDEFGHIKLMNPQRSTVWY')
    return [''.join(rng.choice(aas, size=rng.integers(min_len, max_len))) for _ in range(n_sequences)]
sequences = load_from_huggingface(config['n_sequences'], config['min_length'], config['max_length'])
if sequences is None:
    print("Falling back to synthetic proteins")
    sequences = generate_fallback_proteins(config['n_sequences'], config['min_length'], config['max_length'])
print(f"Sequences: {len(sequences)}, Length range: {min(len(s) for s in sequences)}-{max(len(s) for s in sequences)}")

In [ ]:
# Protein Perturbation Suite
from dataclasses import dataclass, field
AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')
@dataclass
class PerturbedSet:
    name: str; category: str; sequences: list
    params: dict = field(default_factory=dict); description: str = ''
def mutate_protein(sequence, mutation_rate, rng):
    seq = list(sequence); n = max(1, int(len(seq) * mutation_rate))
    for pos in rng.choice(len(seq), size=n, replace=False):
        seq[pos] = rng.choice([aa for aa in AMINO_ACIDS if aa != seq[pos]])
    return ''.join(seq)
class ProteinPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_reverse=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_reverse = include_reverse
    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"aa_sub_{int(rate*100)}pct"
            results[name] = PerturbedSet(name=name, category='substitution',
                sequences=[mutate_protein(s, rate, self.rng) for s in sequences],
                params={'mutation_rate': rate})
        if self.include_reverse:
            results['reverse'] = PerturbedSet(name='reverse', category='reverse',
                sequences=[s[::-1] for s in sequences])
        return results
suite = ProteinPerturbationSuite(seed=320, snp_rates=config['snp_rates'])
print("Perturbation suite ready")

In [ ]:
# SaProt Model Wrapper
#
# SaProt uses a structure-aware vocabulary: each residue is paired with
# a Foldseek 3Di structure token. For sequence-only input (no structure),
# we use '#' as the structure token for every position.
# Example: "MEV" -> "M#E#V#"

import torch
import gc
import numpy as np
from transformers import EsmTokenizer, EsmForMaskedLM


def aa_to_saprot_format(seq):
    """Convert plain AA sequence to SaProt format with '#' structure tokens."""
    return "".join(aa + "#" for aa in seq)


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU memory after cleanup: {mem:.2f} GB")


def make_saprot_embedding_fn(model_name, batch_size=8):
    """Load SaProt and return embedding function."""
    print(f"Loading {model_name}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Device: {device}")

    tokenizer = EsmTokenizer.from_pretrained(model_name)
    model = EsmForMaskedLM.from_pretrained(model_name).to(device).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB")
    else:
        print(f"  Params: {n_params:.1f}M")

    max_len = getattr(model.config, 'max_position_embeddings', 1026)
    print(f"  Max position embeddings: {max_len}")

    # Verify tokenizer works with SaProt format
    test_sa = "M#E#V#"
    test_tokens = tokenizer.tokenize(test_sa)
    print(f"  Tokenizer check: '{test_sa}' -> {test_tokens}")

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size

        for i in range(0, len(sequences), batch_size):
            batch_seqs = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1

            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f"    Batch {batch_idx}/{n_batches}", end='\r')

            # Convert plain AA sequences to SaProt format
            sa_seqs = [aa_to_saprot_format(s) for s in batch_seqs]

            tokens = tokenizer(
                sa_seqs, return_tensors='pt', padding=True,
                truncation=True, max_length=max_len,
            )
            tokens = {key: v.to(device) for key, v in tokens.items()}

            outputs = model(**tokens, output_hidden_states=True)
            last_hidden = outputs.hidden_states[-1]

            attention_mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (last_hidden * attention_mask).sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)
            embeddings.append(pooled.cpu().numpy())

        print()
        return np.concatenate(embeddings, axis=0)

    return embed, model, tokenizer, n_params

print("SaProt wrapper ready")

In [ ]:
# Evaluation Harness
from evaluation_harness import StabilityHarness
harness = StabilityHarness(window_size=0, metric='cosine', n_splits=30, seed=320, max_samples=2500, n_bootstrap=config['n_bootstrap'])
print("Harness configured")

In [ ]:
# Run Experiment
import os, time, pandas as pd
from evaluation_harness import ModelReport
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)
print('=' * 70)
print(f'SAPROT + UNIREF50 -- Phase: {PHASE.upper()}')
print('=' * 70)
reports = []; model_param_counts = []; all_detailed_rows = []
batch_overrides = config.get('batch_overrides', {})
for model_idx, model_name in enumerate(config['models']):
    print(f"\n{'='*70}\n[{model_idx+1}/{len(config['models'])}] {model_name}\n{'='*70}")
    start_time = time.time()
    bs = batch_overrides.get(model_name, config['batch_size'])
    embed_fn, model_obj, tokenizer_obj, n_params_m = make_saprot_embedding_fn(model_name, bs)
    model_param_counts.append(n_params_m)
    perturbed_sets = suite.run_all(sequences)
    safe_name = model_name.replace('/', '_').replace('-', '_') + '_uniref'
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'
    if os.path.exists(cache_clean): embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings...')
        embeddings_clean = embed_fn(sequences); np.save(cache_clean, embeddings_clean)
    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert): perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])
    del model_obj, tokenizer_obj; cleanup_gpu()
    report = harness.evaluate_all_perturbations(model_name=model_name, embeddings_clean=embeddings_clean, perturbed_dict=perturbed_embeddings)
    reports.append(report)
    short_name = model_name.split('/')[-1]
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({'model': short_name, 'size_M': round(n_params_m, 1), 'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0), 'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0), 'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0)})
    print(f'Completed in {(time.time()-start_time)/60:.1f} min | Composite: {report.summary()["mean_composite_stability"]:.4f}')
df_detailed = pd.DataFrame(all_detailed_rows)
df_detailed.to_csv(f'{RESULTS_DIR}/saprot_uniref_{PHASE}_detailed.csv', index=False)
print(f'\nSaved. EXPERIMENT COMPLETE.')

In [ ]:
# Visualization -- Scaling + Cross-Architecture
import matplotlib.pyplot as plt, pandas as pd, glob
from evaluation_harness import compare_models
comparison = compare_models(reports, output_dir=RESULTS_DIR)
composites = [r.summary()['mean_composite_stability'] for r in reports]
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(model_param_counts, composites, 'g^-', linewidth=2, markersize=14, label='SaProt (UniRef50)')
for sz, comp in zip(model_param_counts, composites):
    ax.annotate(f'{comp:.4f}', (sz, comp), textcoords='offset points', xytext=(0,12), ha='center')
base = f'./results'
for label, pattern, color, marker in [
    ('ESM-2 (UniRef)', f'{base}/esm2_*uniref*/results/*_detailed.csv', 'tab:red', 'o-'),
    ('ProtMamba (UniRef)', f'{base}/protmamba_uniref_*/results/*_detailed.csv', 'tab:purple', '*-'),
]:
    files = glob.glob(pattern)
    if files:
        df = pd.read_csv(files[0])
        agg = df.groupby('size_M')['composite'].mean().reset_index().sort_values('size_M')
        ax.plot(agg['size_M'], agg['composite'], marker, color=color, linewidth=2, markersize=10, label=label)
ax.set_xscale('log'); ax.set_xlabel('Parameters (M)'); ax.set_ylabel('Composite Stability')
ax.set_title('SaProt on Real Proteins: Scaling + Cross-Architecture', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/saprot_uniref_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()